# Iteris — Week 1: CAMUS Ingestion + Attention Residual U-Net Baseline
**Project:** DRL-based Automated Medical Image Segmentation  
**Dataset:** CAMUS (Cardiac Acquisitions for Multi-structure Ultrasound Segmentation)  
**Target:** Dice ≥ 0.85 on LV Endocardium  

### Setup on Kaggle:
1. Add dataset: `xiaoweixumedicalai/camus` in the Data panel
2. Enable GPU (T4 x2 or P100)
3. Run all cells

### To switch to CHAOS: change `CFG = CAMUS_CFG` → `CFG = CHAOS_CFG`. Nothing else changes.

## 0 · Install & Imports

In [ ]:
%%capture
!pip install monai[all] -q

In [ ]:
import os, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import monai
from monai.data import Dataset, CacheDataset
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, EnsureTyped,
    Orientationd, Spacingd, Resized,
    ScaleIntensityd, NormalizeIntensityd, ScaleIntensityRanged,
    RandFlipd, RandRotate90d, RandShiftIntensityd,
    RandAffined, RandGaussianNoised,
)
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.utils import set_determinism

print('MONAI:', monai.__version__)
print('PyTorch:', torch.__version__)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1 · Configuration
> Change `CFG = CAMUS_CFG` ↔ `CFG = CHAOS_CFG` to switch datasets. Model and training loop are identical.

**HU window note (CHAOS CT):** `W:150 L:70` → `[L - W/2, L + W/2]` = `[-5, 145]`. This is the standard liver soft-tissue window per the project spec.

In [ ]:
CAMUS_CFG = dict(
    dataset      = 'CAMUS',
    modality     = 'ultrasound',
    data_root    = '/kaggle/input/camus/training',
    image_size   = (256, 256),
    num_classes  = 4,            # 0=bg  1=LV_endo  2=LV_epi  3=LA
    class_names  = ['background', 'LV_endo', 'LV_epi', 'LA'],
    class_colors = ['#000000', '#00C9A7', '#F59E0B', '#F87171'],
    views        = ['2CH', '4CH'],
    phases       = ['ED', 'ES'],
    spacing      = (1.0, 1.0),
    normalize    = 'minmax',
    label_frac   = 1.0,          # 0.1 / 0.2 / 1.0 for few-shot ablations
    val_split    = 0.15,
    test_split   = 0.10,
    batch_size   = 8,
    epochs       = 60,
    lr           = 1e-4,
    weight_decay = 1e-5,
    patience     = 10,
    seed         = 42,
    checkpoint   = '/kaggle/working/camus_best.pt',
    scores_csv   = '/kaggle/working/camus_test_scores.csv',
    masks_dir    = '/kaggle/working/camus_pred_masks',
)

CHAOS_CFG = dict(
    dataset      = 'CHAOS',
    modality     = 'ct',
    data_root    = '/kaggle/input/chaos-dataset',
    image_size   = (256, 256),
    num_classes  = 5,            # 0=bg  1=liver  2=r_kidney  3=l_kidney  4=spleen
    class_names  = ['background', 'liver', 'right_kidney', 'left_kidney', 'spleen'],
    class_colors = ['#000000', '#818CF8', '#34D399', '#86EFAC', '#FB923C'],
    spacing      = (1.5, 1.5),
    normalize    = 'hu',
    # W:150 L:70 → [70 - 75, 70 + 75] = [-5, 145]  (standard liver soft-tissue window)
    hu_window    = (-5, 145),
    label_frac   = 1.0,
    val_split    = 0.15,
    test_split   = 0.10,
    batch_size   = 8,
    epochs       = 60,
    lr           = 1e-4,
    weight_decay = 1e-5,
    patience     = 10,
    seed         = 42,
    checkpoint   = '/kaggle/working/chaos_best.pt',
    scores_csv   = '/kaggle/working/chaos_test_scores.csv',
    masks_dir    = '/kaggle/working/chaos_pred_masks',
)

# ─── ACTIVE CONFIG ───────────────────────────────────────────────────────────
CFG = CAMUS_CFG
# ─────────────────────────────────────────────────────────────────────────────

set_determinism(CFG['seed'])
print(f"Running: {CFG['dataset']} | {CFG['num_classes']} classes | {CFG['image_size']}")

## 2 · Ingestion — Build file-list dicts
MONAI transforms expect `[{'image': path, 'label': path, ...}, ...]`.  
All loading and preprocessing is deferred to the transform pipeline.

In [ ]:
def build_camus_dicts(data_root, views, phases):
    """
    Walks CAMUS dataset and builds {image, label, patient, view, phase} dicts.
    Auto-detects file extension (.nii / .nii.gz / .mhd) and descends into a
    wrapper folder if patients aren't directly under data_root.
    """
    root = Path(data_root)

    # Some Kaggle uploads add a wrapper dir (e.g. .../camus/CAMUS/patient0001).
    # If no patient* folders here, descend one level.
    def has_patients(p):
        return any(c.is_dir() and c.name.lower().startswith('patient') for c in p.iterdir())

    if not has_patients(root):
        for sub in root.iterdir():
            if sub.is_dir() and has_patients(sub):
                root = sub
                break
    print(f'Walking: {root}')

    EXTS = ['.nii.gz', '.nii', '.mhd']
    records, missing = [], []
    for patient_dir in sorted(root.iterdir()):
        if not patient_dir.is_dir() or not patient_dir.name.lower().startswith('patient'):
            continue
        pid = patient_dir.name
        for view in views:
            for phase in phases:
                img = lbl = None
                for ext in EXTS:
                    cand_img = patient_dir / f'{pid}_{view}_{phase}{ext}'
                    cand_lbl = patient_dir / f'{pid}_{view}_{phase}_gt{ext}'
                    if cand_img.exists() and cand_lbl.exists():
                        img, lbl = cand_img, cand_lbl
                        break
                if img is not None:
                    records.append(dict(
                        image=str(img), label=str(lbl),
                        patient=pid, view=view, phase=phase,
                    ))
                else:
                    missing.append(f'{pid}_{view}_{phase}')

    if not records:
        print(f'  (no matches found — first 5 entries in {root}):')
        for p in sorted(root.iterdir())[:5]:
            print(f'    {p.name}')
    elif missing:
        print(f'  Skipped {len(missing)} missing combos (first 3): {missing[:3]}')
    return records


def build_chaos_ct_dicts(data_root):
    """CHAOS CT: DICOM slices + PNG masks. Inactive until CFG = CHAOS_CFG."""
    import pydicom
    from PIL import Image as PILImage
    LABEL_MAP = {55: 1, 110: 2, 155: 3, 200: 4}
    root, records = Path(data_root) / 'CT', []
    for patient_dir in sorted(root.iterdir()):
        dcm_dir = patient_dir / 'DICOM_anon'
        gt_dir  = patient_dir / 'Ground'
        if not dcm_dir.exists(): continue
        for dcm_f, gt_f in zip(sorted(dcm_dir.glob('*.dcm')), sorted(gt_dir.glob('*.png'))):
            records.append(dict(
                image=str(dcm_f), label=str(gt_f),
                patient=patient_dir.name,
            ))
    return records


if CFG['dataset'] == 'CAMUS':
    all_dicts = build_camus_dicts(CFG['data_root'], CFG['views'], CFG['phases'])
elif CFG['dataset'] == 'CHAOS':
    all_dicts = build_chaos_ct_dicts(CFG['data_root'])

print(f'\nTotal file pairs: {len(all_dicts)}')
if all_dicts:
    print('Example:', {k: v for k, v in all_dicts[0].items() if k in ('image','label','patient')})
else:
    raise RuntimeError(
        f'No file pairs found under {CFG["data_root"]}. '
        f'Check that the dataset is mounted and the path is correct.'
    )

## 3 · MONAI Transform Pipelines

**Transform order matters:** `Spacingd` (resample to isotropic spacing) must come before `Resized`  
(resize to fixed dims). Reversing them causes inconsistent scale distortion across patients.

**`Orientationd` note:** `axcodes='RAS'` is a no-op for CAMUS 2D `.mhd` slices (singleton z).  
Kept for forward-compatibility with ACDC 3D volumes. For ACDC, verify scanner orientation  
and apply the 180° flip noted in the project spec before this transform.

In [ ]:
def build_intensity_transform(cfg):
    if cfg['normalize'] == 'minmax':
        return ScaleIntensityd(keys=['image'], minv=0.0, maxv=1.0)
    if cfg['normalize'] == 'zscore':
        return NormalizeIntensityd(keys=['image'], nonzero=True, channel_wise=True)
    if cfg['normalize'] == 'hu':
        a_min, a_max = cfg['hu_window']
        return ScaleIntensityRanged(
            keys=['image'], a_min=a_min, a_max=a_max,
            b_min=0.0, b_max=1.0, clip=True,
        )
    raise ValueError(f"Unknown normalize: {cfg['normalize']}")


def make_transforms(cfg, split='train'):
    base = [
        LoadImaged(keys=['image', 'label']),
        EnsureChannelFirstd(keys=['image', 'label']),
        Orientationd(keys=['image', 'label'], axcodes='RAS'),
        Spacingd(
            keys=['image', 'label'],
            pixdim=(*cfg['spacing'], -1),
            mode=('bilinear', 'nearest'),
        ),
        Resized(
            keys=['image', 'label'],
            spatial_size=cfg['image_size'],
            mode=('bilinear', 'nearest'),
        ),
        build_intensity_transform(cfg),
        EnsureTyped(keys=['image', 'label'], dtype=(torch.float32, torch.long)),
    ]
    aug = [
        RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=1),
        RandRotate90d(keys=['image', 'label'], prob=0.3, max_k=3),
        RandAffined(
            keys=['image', 'label'], prob=0.6,
            rotate_range=(0.26,), scale_range=(0.1,), translate_range=(10,),
            mode=('bilinear', 'nearest'), padding_mode='border',
        ),
        RandShiftIntensityd(keys=['image'], offsets=0.15, prob=0.4),
        RandGaussianNoised(keys=['image'], prob=0.3, std=0.02),
    ] if split == 'train' else []
    return Compose(base + aug)


train_tfm = make_transforms(CFG, split='train')
eval_tfm  = make_transforms(CFG, split='val')
print('Transform pipelines built.')

## 4 · Patient-level Split

**`label_frac` is applied at patient level** — all views/phases of a selected patient are  
kept together. Applying it at file level would split 2CH/4CH of the same patient, which  
would corrupt the few-shot ablation curves in Week 4.

In [ ]:
# All unique patients, shuffled with fixed seed
all_patients = sorted({d['patient'] for d in all_dicts})
rng = random.Random(CFG['seed'])
rng.shuffle(all_patients)

# ─── Label-fraction sub-sampling at patient level ─────────────────────────────
# Applied BEFORE split so val/test always use full data; only train shrinks.
n_test  = max(1, int(len(all_patients) * CFG['test_split']))
n_val   = max(1, int(len(all_patients) * CFG['val_split']))
test_pts  = set(all_patients[:n_test])
val_pts   = set(all_patients[n_test : n_test + n_val])
train_pts_full = all_patients[n_test + n_val:]

if CFG['label_frac'] < 1.0:
    n_keep    = max(1, int(len(train_pts_full) * CFG['label_frac']))
    train_pts = set(train_pts_full[:n_keep])   # deterministic: first N in shuffled list
    print(f'Few-shot: using {n_keep}/{len(train_pts_full)} training patients ({CFG["label_frac"]*100:.0f}%)')
else:
    train_pts = set(train_pts_full)

train_dicts = [d for d in all_dicts if d['patient'] in train_pts]
val_dicts   = [d for d in all_dicts if d['patient'] in val_pts]
test_dicts  = [d for d in all_dicts if d['patient'] in test_pts]
print(f'Patients  — Train: {len(train_pts)}  Val: {len(val_pts)}  Test: {len(test_pts)}')
print(f'Samples   — Train: {len(train_dicts)}  Val: {len(val_dicts)}  Test: {len(test_dicts)}')

train_ds = CacheDataset(train_dicts, transform=train_tfm, cache_rate=1.0, num_workers=2)
val_ds   = CacheDataset(val_dicts,   transform=eval_tfm,  cache_rate=1.0, num_workers=2)
test_ds  = Dataset(test_dicts,       transform=eval_tfm)

train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=1,                shuffle=False, num_workers=2, pin_memory=True)

batch = next(iter(train_dl))
print(f'\nBatch — image: {batch["image"].shape}  label: {batch["label"].shape}')

## 5 · EDA — Sanity Check

In [ ]:
from matplotlib.colors import ListedColormap
cmap = ListedColormap(CFG['class_colors'])

fig, axes = plt.subplots(4, 2, figsize=(8, 16))
for row in range(4):
    img  = batch['image'][row, 0].numpy()
    mask = batch['label'][row, 0].numpy()
    axes[row,0].imshow(img, cmap='gray'); axes[row,0].set_title('Image');   axes[row,0].axis('off')
    axes[row,1].imshow(img, cmap='gray')
    axes[row,1].imshow(mask, cmap=cmap, alpha=0.5, vmin=0, vmax=CFG['num_classes']-1)
    axes[row,1].set_title('Overlay'); axes[row,1].axis('off')

patches = [mpatches.Patch(color=c, label=n)
           for c, n in zip(CFG['class_colors'][1:], CFG['class_names'][1:])]
fig.legend(handles=patches, loc='upper right')
plt.suptitle(f'{CFG["dataset"]} — Train batch sample')
plt.tight_layout(); plt.show()

## 6 · Model — Attention Residual U-Net
Custom implementation for paper reproducibility. `in_channels` and `num_classes` are the  
only parameters that differ between datasets.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.skip  = (
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch))
            if in_ch != out_ch else nn.Identity()
        )
    def forward(self, x):
        return self.relu(self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))) + self.skip(x))


class AttentionGate(nn.Module):
    """Soft attention gate — Oktay et al. (2018)."""
    def __init__(self, F_g, F_x, F_int):
        super().__init__()
        self.W_g  = nn.Sequential(nn.Conv2d(F_g,   F_int, 1, bias=False), nn.BatchNorm2d(F_int))
        self.W_x  = nn.Sequential(nn.Conv2d(F_x,   F_int, 1, bias=False), nn.BatchNorm2d(F_int))
        self.psi  = nn.Sequential(nn.Conv2d(F_int, 1,     1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        g_up  = F.interpolate(self.W_g(g), size=x.shape[2:], mode='bilinear', align_corners=False)
        return x * self.psi(self.relu(g_up + self.W_x(x)))


class AttentionResUNet(nn.Module):
    """
    Attention Residual U-Net — dataset-agnostic.
    in_channels : 1 for greyscale (CAMUS / CHAOS / ACDC / DRIVE / ISIC)
                  4 for BraTS multi-parametric MRI
    num_classes : from CFG['num_classes']
    """
    def __init__(self, in_channels=1, num_classes=4, features=(64, 128, 256, 512)):
        super().__init__()
        F = features
        self.pool       = nn.MaxPool2d(2)
        self.enc1       = ResBlock(in_channels, F[0])
        self.enc2       = ResBlock(F[0], F[1])
        self.enc3       = ResBlock(F[1], F[2])
        self.enc4       = ResBlock(F[2], F[3])
        self.bottleneck = ResBlock(F[3], F[3]*2)
        self.att4 = AttentionGate(F[3]*2, F[3], F[3]//2)
        self.att3 = AttentionGate(F[3],   F[2], F[2]//2)
        self.att2 = AttentionGate(F[2],   F[1], F[1]//2)
        self.att1 = AttentionGate(F[1],   F[0], F[0]//2)
        self.up4  = nn.ConvTranspose2d(F[3]*2, F[3], 2, stride=2)
        self.dec4 = ResBlock(F[3]*2, F[3])
        self.up3  = nn.ConvTranspose2d(F[3], F[2], 2, stride=2)
        self.dec3 = ResBlock(F[2]*2, F[2])
        self.up2  = nn.ConvTranspose2d(F[2], F[1], 2, stride=2)
        self.dec2 = ResBlock(F[1]*2, F[1])
        self.up1  = nn.ConvTranspose2d(F[1], F[0], 2, stride=2)
        self.dec1 = ResBlock(F[0]*2, F[0])
        self.head = nn.Conv2d(F[0], num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b),  self.att4(b,  e4)], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), self.att3(d4, e3)], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), self.att2(d3, e2)], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), self.att1(d2, e1)], 1))
        return self.head(d1)


model = AttentionResUNet(in_channels=1, num_classes=CFG['num_classes']).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params/1e6:.2f}M')
with torch.no_grad():
    sz  = CFG['image_size']
    out = model(torch.zeros(2, 1, sz[0], sz[1]).to(DEVICE))
print(f'Output: {out.shape}  (expect [2, {CFG["num_classes"]}, {sz[0]}, {sz[1]}])')

## 7 · Loss & Metrics (MONAI)
Dice, IoU, HD, and 95HD are all computed — the full set required by IEEE JBHI.

In [ ]:
criterion = DiceCELoss(
    to_onehot_y=True, softmax=True,
    include_background=False,
    lambda_dice=0.5, lambda_ce=0.5,
)

# Val-time metrics (mean_batch → per-class means across the batch)
dice_metric = DiceMetric(
    include_background=False, reduction='mean_batch', get_not_nans=True,
)

# Test-time metrics — per-sample, aggregated manually so we keep per-patient rows
dice_metric_test = DiceMetric(
    include_background=False, reduction='none', get_not_nans=True,
)
hd95_metric_test = HausdorffDistanceMetric(
    include_background=False, percentile=95, reduction='none', get_not_nans=True,
)

def to_onehot(t, num_classes):
    """(B,1,H,W) int → (B,C,H,W) float one-hot."""
    return F.one_hot(t.squeeze(1).long(), num_classes).permute(0,3,1,2).float()

print('Loss and metrics ready.')

## 8 · Training Loop
Logs both `train_loss` and `train_dice` — needed for the convergence figure (Figure 6).

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-6)


def train_epoch(model, loader, optimizer, criterion, device, num_classes):
    model.train()
    total_loss  = 0.0
    dice_vals   = []
    for batch in loader:
        imgs   = batch['image'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        # Compute batch-level Dice without gradient (no extra forward pass)
        with torch.no_grad():
            pred_oh  = to_onehot(logits.argmax(1, keepdim=True), num_classes)
            label_oh = to_onehot(labels, num_classes)
            inter = (pred_oh[:,1:] * label_oh[:,1:]).sum(dim=(0,2,3))
            denom = (pred_oh[:,1:] + label_oh[:,1:]).sum(dim=(0,2,3))
            dice_vals.append(((2*inter + 1e-6) / (denom + 1e-6)).mean().item())
    return total_loss / len(loader), float(np.mean(dice_vals))


@torch.no_grad()
def eval_epoch(model, loader, criterion, device, num_classes, dice_metric):
    model.eval()
    total_loss = 0.0
    dice_metric.reset()
    for batch in loader:
        imgs, labels = batch['image'].to(device), batch['label'].to(device)
        logits        = model(imgs)
        total_loss   += criterion(logits, labels).item()
        pred_oh       = to_onehot(logits.argmax(1, keepdim=True), num_classes)
        label_oh      = to_onehot(labels, num_classes)
        dice_metric(y_pred=pred_oh, y=label_oh)
    per_class_dice, _ = dice_metric.aggregate()
    dice_metric.reset()
    return total_loss / len(loader), per_class_dice.cpu().numpy()


# ─── Training ─────────────────────────────────────────────────────────────────
best_dice   = 0.0
patience_ct = 0
history     = {'train_loss': [], 'train_dice': [], 'val_loss': [], 'val_dice_mean': []}

print(f'Training {CFG["dataset"]} | {CFG["epochs"]} epochs | {DEVICE}')
for epoch in range(1, CFG['epochs'] + 1):
    tr_loss, tr_dice         = train_epoch(model, train_dl, optimizer, criterion, DEVICE, CFG['num_classes'])
    vl_loss, per_class_dice  = eval_epoch(model, val_dl, criterion, DEVICE, CFG['num_classes'], dice_metric)
    scheduler.step()

    mean_dice = float(np.nanmean(per_class_dice))
    history['train_loss'].append(tr_loss)
    history['train_dice'].append(tr_dice)
    history['val_loss'].append(vl_loss)
    history['val_dice_mean'].append(mean_dice)

    if mean_dice > best_dice:
        best_dice, patience_ct = mean_dice, 0
        torch.save(model.state_dict(), CFG['checkpoint'])
        marker = '✓'
    else:
        patience_ct += 1
        marker = ' '

    class_str = '  '.join(f'{CFG["class_names"][c+1]}:{per_class_dice[c]:.3f}'
                          for c in range(len(per_class_dice)))
    print(f'Ep {epoch:03d} | tr_loss {tr_loss:.4f} tr_dice {tr_dice:.4f} | '
          f'vl_loss {vl_loss:.4f} Dice {mean_dice:.4f} {marker} | {class_str}')

    if patience_ct >= CFG['patience']:
        print(f'Early stop at epoch {epoch}.')
        break

print(f'\nBest val Dice: {best_dice:.4f}')

## 9 · Learning Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(history['train_loss'], label='Train loss')
ax1.plot(history['val_loss'],   label='Val loss')
ax1.set(xlabel='Epoch', ylabel='Loss', title='Loss curves'); ax1.legend()

ax2.plot(history['train_dice'],     label='Train Dice', linestyle='--', alpha=0.7)
ax2.plot(history['val_dice_mean'],  label='Val Dice')
ax2.axhline(0.85, color='red', linestyle='--', label='Target 0.85')
ax2.set(xlabel='Epoch', ylabel='Dice', title=f'{CFG["dataset"]} — Dice curves'); ax2.legend()

plt.suptitle(f'{CFG["dataset"]} Attention Residual U-Net — Week 1 Baseline')
plt.tight_layout()
plt.savefig('/kaggle/working/learning_curves.png', dpi=150)
plt.show()

## 10 · Test-set Evaluation — Full Metrics

Computes **Dice, 95HD per structure per sample** and saves a per-patient CSV.  
This CSV is consumed by the Week 4 Wilcoxon signed-rank tests and by the DRL  
pipeline in Week 2 (per-image baseline Dice for reward initialisation).

In [ ]:
model.load_state_dict(torch.load(CFG['checkpoint'], map_location=DEVICE))
model.eval()

dice_metric_test.reset()
hd95_metric_test.reset()
rows = []

with torch.no_grad():
    for i, batch in enumerate(tqdm(test_dl, desc='Test eval')):
        imgs   = batch['image'].to(DEVICE)   # (1, 1, H, W)
        labels = batch['label'].to(DEVICE)   # (1, 1, H, W)
        logits = model(imgs)
        pred_oh  = to_onehot(logits.argmax(1, keepdim=True), CFG['num_classes'])
        label_oh = to_onehot(labels, CFG['num_classes'])

        dice_metric_test(y_pred=pred_oh, y=label_oh)   # accumulates per-sample
        hd95_metric_test(y_pred=pred_oh, y=label_oh)

        # Build a row for the CSV (batch_size=1 → index 0)
        dice_vals, _ = dice_metric_test.aggregate()  # (1, C-1)
        hd95_vals, _ = hd95_metric_test.aggregate()
        dice_metric_test.reset()
        hd95_metric_test.reset()

        row = {
            'patient': batch['patient'][0],
            'view':    batch.get('view',  [''])[0],
            'phase':   batch.get('phase', [''])[0],
        }
        for c, name in enumerate(CFG['class_names'][1:]):
            row[f'dice_{name}'] = float(dice_vals[0, c]) if not torch.isnan(dice_vals[0, c]) else np.nan
            row[f'hd95_{name}'] = float(hd95_vals[0, c]) if not torch.isnan(hd95_vals[0, c]) else np.nan
        rows.append(row)

scores_df = pd.DataFrame(rows)
scores_df.to_csv(CFG['scores_csv'], index=False)
print(f'Saved per-sample scores → {CFG["scores_csv"]}')
print(scores_df.head())

# Summary
print('\n── Test Results ────────────────────────────────────────────────')
for name in CFG['class_names'][1:]:
    d = scores_df[f'dice_{name}'].mean()
    h = scores_df[f'hd95_{name}'].mean()
    print(f'  {name:15s}:  Dice {d:.4f}  |  95HD {h:.2f} mm')
print(f'  {"Mean":15s}:  Dice {scores_df[[c for c in scores_df if c.startswith("dice_")]].mean().mean():.4f}')
print('────────────────────────────────────────────────────────────────')
print(f'  LV_endo Dice ≥ 0.85: {"PASS ✓" if scores_df["dice_LV_endo"].mean() >= 0.85 else "not yet"}')

## 11 · Save Predicted Masks
Exports test-set predicted masks as `.npy` files.  
The Week 2 DRL environment loads these as the initial boundary state (`mask_0`) for each episode.

In [ ]:
import os
os.makedirs(CFG['masks_dir'], exist_ok=True)

model.eval()
with torch.no_grad():
    for batch in tqdm(test_dl, desc='Exporting masks'):
        imgs   = batch['image'].to(DEVICE)
        pred   = model(imgs).argmax(1).squeeze(0).cpu().numpy().astype(np.uint8)  # (H, W)
        pid    = batch['patient'][0]
        view   = batch.get('view',  [''])[0]
        phase  = batch.get('phase', [''])[0]
        fname  = f'{pid}_{view}_{phase}_pred.npy' if view else f'{pid}_pred.npy'
        np.save(os.path.join(CFG['masks_dir'], fname), pred)

print(f'Masks saved to {CFG["masks_dir"]}/')

## 12 · Qualitative Visualisation

In [ ]:
cmap   = ListedColormap(CFG['class_colors'])
n_show = 4
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4*n_show))

model.eval()
for row, batch in enumerate(test_dl):
    if row >= n_show: break
    img_t  = batch['image'][0].to(DEVICE)
    label  = batch['label'][0, 0].numpy()
    with torch.no_grad():
        pred = model(img_t.unsqueeze(0)).argmax(1)[0].cpu().numpy()
    img_np = img_t[0].cpu().numpy()

    axes[row,0].imshow(img_np, cmap='gray'); axes[row,0].set_title('Input');       axes[row,0].axis('off')
    axes[row,1].imshow(img_np, cmap='gray')
    axes[row,1].imshow(label, cmap=cmap, alpha=0.5, vmin=0, vmax=CFG['num_classes']-1)
    axes[row,1].set_title('Ground Truth'); axes[row,1].axis('off')
    axes[row,2].imshow(img_np, cmap='gray')
    axes[row,2].imshow(pred, cmap=cmap, alpha=0.5, vmin=0, vmax=CFG['num_classes']-1)
    axes[row,2].set_title('Prediction');   axes[row,2].axis('off')

patches = [mpatches.Patch(color=c, label=n)
           for c, n in zip(CFG['class_colors'][1:], CFG['class_names'][1:])]
fig.legend(handles=patches, loc='upper right')
plt.suptitle(f'{CFG["dataset"]} — Qualitative Results')
plt.tight_layout()
plt.savefig('/kaggle/working/qualitative_results.png', dpi=150)
plt.show()

## 13 · 5-Fold CV Skeleton
The 5-fold loop is pre-wired with fixed seeds so results are reproducible for Week 4 stats.  
**Not run by default** (set `RUN_CV = True` to activate). Consumes ~5× the training time.

In [ ]:
RUN_CV = False   # set True in Week 4

if RUN_CV:
    from sklearn.model_selection import KFold

    # Use only non-test patients for CV (test set is held out entirely)
    cv_patients = sorted(train_pts | val_pts)
    kf = KFold(n_splits=5, shuffle=True, random_state=CFG['seed'])
    cv_results  = []

    for fold, (tr_idx, vl_idx) in enumerate(kf.split(cv_patients)):
        fold_train = {cv_patients[i] for i in tr_idx}
        fold_val   = {cv_patients[i] for i in vl_idx}

        fold_train_d = [d for d in all_dicts if d['patient'] in fold_train]
        fold_val_d   = [d for d in all_dicts if d['patient'] in fold_val]

        fold_model = AttentionResUNet(
            in_channels=1, num_classes=CFG['num_classes']
        ).to(DEVICE)
        fold_opt = torch.optim.AdamW(
            fold_model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay']
        )

        f_train_dl = DataLoader(
            CacheDataset(fold_train_d, transform=train_tfm, cache_rate=1.0),
            batch_size=CFG['batch_size'], shuffle=True, num_workers=2,
        )
        f_val_dl = DataLoader(
            CacheDataset(fold_val_d, transform=eval_tfm, cache_rate=1.0),
            batch_size=CFG['batch_size'], shuffle=False, num_workers=2,
        )

        fold_dice_metric = DiceMetric(
            include_background=False, reduction='mean_batch', get_not_nans=True,
        )
        fold_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            fold_opt, T_max=CFG['epochs'], eta_min=1e-6
        )

        best_fold_dice = 0.0
        for epoch in range(1, CFG['epochs'] + 1):
            train_epoch(fold_model, f_train_dl, fold_opt, criterion, DEVICE, CFG['num_classes'])
            _, fold_dice = eval_epoch(
                fold_model, f_val_dl, criterion, DEVICE, CFG['num_classes'], fold_dice_metric
            )
            fold_scheduler.step()
            best_fold_dice = max(best_fold_dice, float(np.nanmean(fold_dice)))

        cv_results.append({'fold': fold+1, 'val_dice': best_fold_dice})
        print(f'Fold {fold+1}: best val Dice = {best_fold_dice:.4f}')

    cv_df = pd.DataFrame(cv_results)
    print(f'\n5-fold CV Dice: {cv_df["val_dice"].mean():.4f} ± {cv_df["val_dice"].std():.4f}')
    cv_df.to_csv('/kaggle/working/cv_results.csv', index=False)
else:
    print('CV skipped (RUN_CV=False). Enable in Week 4 for significance tests.')

## 14 · Save Summary JSON

In [ ]:
test_dice_means = {name: float(scores_df[f'dice_{name}'].mean())
                   for name in CFG['class_names'][1:]}
test_hd95_means = {name: float(scores_df[f'hd95_{name}'].mean())
                   for name in CFG['class_names'][1:]}

summary = dict(
    dataset        = CFG['dataset'],
    model          = 'AttentionResUNet',
    num_classes    = CFG['num_classes'],
    image_size     = CFG['image_size'],
    checkpoint     = CFG['checkpoint'],
    masks_dir      = CFG['masks_dir'],
    scores_csv     = CFG['scores_csv'],
    best_val_dice  = round(float(best_dice), 4),
    test_dice      = {k: round(v, 4) for k, v in test_dice_means.items()},
    test_dice_mean = round(float(np.mean(list(test_dice_means.values()))), 4),
    test_hd95      = {k: round(v, 2) for k, v in test_hd95_means.items()},
)
out = CFG['checkpoint'].replace('.pt', '_summary.json')
with open(out, 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))